In [7]:
import pandas as pd

pd.set_option('display.max_columns', None)#打印时显示所有列。

# 从CSV文件读取数据（确保你有正确的路径）
df = pd.read_csv(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')
print(df.head(5))

# 去除不需要的列
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# 去除 Age 缺失的样本
df = df.dropna(subset=["Age"])

# 对 Sex 和 Embarked 做独热编码
df = pd.get_dummies(df, columns=["Sex", "Embarked"],dtype=int)

print(df.head(10))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
  

在深度学习项目中，数据处理是非常重要的一环，PyTorch提供这两个类来帮助用户高效地加载和处理数据。Dataset负责数据的读取和预处理，而DataLoader则负责将数据分成小批量，支持多线程加速，以及数据的打乱等。本节，我们就以Titanic数据集为例，讲解如何使用PyTorch里的Dataset和DataLoader类来处理数据。

In [4]:
from torch.utils.data import Dataset
import pandas as pd
import torch


class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910
        }

        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"]) ##删除不用的列
        df = df.dropna(subset=["Age"])##删除Age有缺失的行
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)##进行one-hot编码

        ##进行数据的标准化
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
        for i in range(len(base_features)):
            df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

Dataset类提供对数据集的抽象，任何自定义数据集都需要继承torch.utils.data.Dataset,并实现两个方法：__len__和__getitem__(idx)。其中__len__需要返回整个数据集样本的个数。__getitem__(idx)需要能根据样本的index返回具体的样本。

另外一般情况下，我们也把对数据的预处理工作放在自定义的Dataset类定义里。

一般情况下，我们不需要自定义DataLoader。PyTorch里默认实现的DataLoader就可以满足我们的使用，它定义了如何批量读取数据的功能。比如你可以通过batchsize设置每次读取数据的大小，通过shuffle参数设置是否对数据集进行打乱。另外如果你的Dataset的`_getitem__`比较费时，你可以通过num_workers参数指定多进程加载。

In [6]:
from torch.utils.data import DataLoader

dataset = TitanicDataset(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
for inputs, labels in dataloader:
    print(inputs.shape, labels.shape)
    break

torch.Size([32, 10]) torch.Size([32])


nn.Module是PyTorch里的核心组件，在PyTorch里所有的模型，或者模型内的模块，都是基于nn.Module构建的。假如你定义一个复杂的模型，它是由多个嵌套的模块构成的。那么你的这个复杂模型和内部的各个可训练模块，都必须继承自nn.Module。正是因为所有可训练的模块都是基于nn.Module构建，PyTorch可以方便的对可训练参数进行管理，并为我们的实现提供了便利。

nn.Module带来了以下好处：

模块化构建：对于复杂的网络，你可以将它划分为多个子模块进行构建。然后将它们组合为一个复杂模型。
自动管理参数：nn.Module会自动追踪模块内所有的参数，无论这些参数是嵌套在子模块中还是作为属性值直接存储在当前模块中。可以通过模块的parameters()或者named_parameters()进行查看。
统一的forward方法：所有继承自nn.Module的类必须实现一个forward方法，这样各个模块之间根据组合关系对数据进行前向处理。PyTorch里的计算图和自动求梯度机制会帮助我们实现反向传播。
统一的设备管理：利用nn.Module提供的.to(device)方法，可以方便的将整个模型迁移到GPU或者CPU。
模型的保存和加载：nn.Module提供了标准的模型保存和加载方法。
定义了模型的train和eval状态：有的模块在训练时和预测（或者叫推理）时前向传播实现是不同的，可以通过model.train()或者model.eval()统一切换本身，和内部包含的所有模块的状态。

In [ ]:
#逻辑回归代码展开,这一段不需要运行
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # nn.Linear也继承自nn.Module，输入为input_dim,输出一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))  # Logistic Regression 输出概率

其中nn.Linear是定义一个线性回归模型。它传入两个参数，第一个是输入特征的个数。第二个是输出特征的个数，你可能奇怪线性回归输出的个数不就是1吗，为什么还可以是其他值？实际上这里的nn.Linear是一个线性层。它可以一次性定义多个线性回归，所以可以有多个输出，当我们设置输出值为1时，那么就是我们之前讲的线性回归了。关于nn.Linear的更多知识我们会在讲神经网络时谈及。